## Filter Steering Dataset

### Overview

Filters a prompt set down to the examples where an activation-patching intervention reliably flips the model's answer from factual to counterfactual, based on clean vs. intervened result files. Saves the filtered subset as the dataset used by the other steering notebooks.


In [ ]:
import sys
sys.path.append("../../src")

import pandas as pd

import _config

prompt_config = _config.PromptConfig(model_type="GPT-OSS_stepwise", prompt_type="h_pre_penultimate_sum")
activation_result_dir = "frozen_attention_intervened_pattern"

# Read the data file
data_file = _config.load_prompts(prompt_config)

# Read the clean and intervened results
clean_path = _config.build_run_output_path(
    prompt_config,
    _config.RunConfig(
        experiment_root="experiments/activation_intervention",
        result_dir=activation_result_dir,
        output_filename="non_intervened_pattern_h_pre_penultimate_sum_.csv",
    ),
)
intervened_path = _config.build_run_output_path(
    prompt_config,
    _config.RunConfig(
        experiment_root="experiments/activation_intervention",
        result_dir=activation_result_dir,
        output_filename="h_pre_penultimate_sum_.csv",
    ),
)
clean_results = pd.read_csv(clean_path)
intervened_results = pd.read_csv(intervened_path)

# List to store rows that meet the criteria
filtered_rows = []

# Process each row in the data file
for _, data_row in data_file.iterrows():
    # Find matching rows in clean results
    clean_matches = clean_results[
        (clean_results['base_1_num'] == data_row['base_1_num']) &
        (clean_results['base_2_num'] == data_row['base_2_num']) &
        (clean_results['source_1_num'] == data_row['source_1_num']) &
        (clean_results['source_2_num'] == data_row['source_2_num'])
    ]
    
    # Find matching rows in intervened results
    intervened_matches = intervened_results[
        (intervened_results['base_1_num'] == data_row['base_1_num']) &
        (intervened_results['base_2_num'] == data_row['base_2_num']) &
        (intervened_results['source_1_num'] == data_row['source_1_num']) &
        (intervened_results['source_2_num'] == data_row['source_2_num'])
    ]
    
    # Check if we have exactly 20 matches in each file
    if len(clean_matches) == 20 and len(intervened_matches) == 20:
        # Count how many rows have generated_text equal to counterfactual_output
        clean_count = (clean_matches['generated_text'] == clean_matches['counterfactual_output']).sum()
        intervened_count = (intervened_matches['generated_text'] == intervened_matches['counterfactual_output']).sum()
        
        # Check if clean_count is at least 14 more than intervened_count
        if clean_count >= intervened_count + 14:
            filtered_rows.append(data_row)
    else:
        print(f"Skipping row {data_row['base_1_num']}+{data_row['base_2_num']}, {data_row['source_1_num']}+{data_row['source_2_num']}")

# Create DataFrame from filtered rows
filtered_df = pd.DataFrame(filtered_rows)

# Save the filtered dataset
output_path = _config.build_run_output_path(
    prompt_config,
    _config.RunConfig(
        experiment_root="experiments/steering",
        result_dir="steering_dataset",
        output_filename="filtered_steering_dataset.csv",
    ),
)
output_path.parent.mkdir(parents=True, exist_ok=True)
filtered_df.to_csv(output_path, index=False)

print(f"Filtered dataset saved to {output_path}")
print(f"Original dataset size: {len(data_file)}")
print(f"Filtered dataset size: {len(filtered_df)}")

Filtered dataset saved to ../../experiments/steering/output/GPT-OSS_stepwise/steering_dataset/filtered_steering_dataset.csv
Original dataset size: 256
Filtered dataset size: 120
